# EbbRAG — Ablations (SPEC-05)

Sweeps over EbbRAG's hyperparameters to isolate the contribution of each component.

**Sweeps:** decay rate λ · active-recall weight α3 · retrieval depth k

**Requires:** `EbbRAG_01_core.ipynb` and `EbbRAG_04_data.ipynb` already run in this session

## 1. Install & imports

In [ ]:
!pip install -q anthropic>=0.25.0 sentence-transformers>=2.7.0 matplotlib pandas tqdm

from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

from dataclasses import dataclass, field
from typing import Optional, List
import time, math, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anthropic
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

os.makedirs('results/ablations', exist_ok=True)
print('Imports ✓')

## 2. Minimal core classes (re-defined for standalone ablation runs)

In [ ]:
@dataclass
class Chunk:
    id: str
    text: str
    embedding: Optional[List[float]] = None
    stability: float = 0.5
    last_retrieved: float = field(default_factory=time.time)
    retrieval_count: int = 0

@dataclass
class QueryResult:
    query_id: str
    query: str
    retrieved_chunks: List[Chunk]
    retrieval_scores: List[float]
    parametric_answer: Optional[str]
    final_answer: str
    latency_ms: float
    tokens_used: int
    em: Optional[int] = None

class Embedder:
    def __init__(self, model_name: str = 'BAAI/bge-small-en-v1.5'):
        self.model = SentenceTransformer(model_name)
    def embed(self, text: str) -> np.ndarray:
        return self.model.encode(text, normalize_embeddings=True)
    def embed_batch(self, texts: List[str], batch_size: int = 64) -> np.ndarray:
        return self.model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)

print('Base dataclasses ✓')

In [ ]:
class ChunkIndex:
    """Index with SR-weighted (spaced-repetition) retrieval scoring."""
    def __init__(self, lambda_decay: float = 0.05, epsilon: float = 0.01):
        self.chunks: List[Chunk] = []
        self.embeddings: Optional[np.ndarray] = None
        self.lambda_decay = lambda_decay
        self.epsilon = epsilon

    def add_chunks(self, chunks: List[Chunk], embedder: Embedder):
        embs = embedder.embed_batch([c.text for c in chunks])
        for chunk, emb in zip(chunks, embs):
            chunk.embedding = emb.tolist()
        self.chunks.extend(chunks)
        self.embeddings = np.array([c.embedding for c in self.chunks])

    def sr_weight(self, chunk: Chunk, t_now: float) -> float:
        """Ebbinghaus-style retention weight: decays with elapsed time, scaled by stability."""
        elapsed = max(0.0, t_now - chunk.last_retrieved)
        retention = math.exp(-self.lambda_decay * elapsed / max(chunk.stability, self.epsilon))
        return self.epsilon + (1 - self.epsilon) * retention

    def search(self, query_emb: np.ndarray, k: int = 5, t_now: Optional[float] = None,
               alpha1: float = 1.0, alpha2: float = 0.0,
               candidate_emb: Optional[np.ndarray] = None, alpha3: float = 0.0) -> List[tuple]:
        sim = self.embeddings @ query_emb
        if t_now is None:
            mixed = sim
        else:
            sr = np.array([self.sr_weight(c, t_now) for c in self.chunks])
            mixed = alpha1 * sim + alpha2 * sr
            if candidate_emb is not None and alpha3 > 0:
                ar_sim = self.embeddings @ candidate_emb
                mixed = mixed + alpha3 * ar_sim
        top_idx = np.argsort(mixed)[::-1][:k]
        return [(self.chunks[i], float(mixed[i])) for i in top_idx]

    def update_sr(self, chunk: Chunk, t_now: float):
        """Reinforce a chunk's stability after retrieval (spacing-effect update)."""
        elapsed = max(0.0, t_now - chunk.last_retrieved)
        chunk.stability = chunk.stability + math.log1p(elapsed / 86400.0)
        chunk.last_retrieved = t_now
        chunk.retrieval_count += 1

    @classmethod
    def load(cls, path: str) -> 'ChunkIndex':
        with open(path, 'rb') as f: return pickle.load(f)

print('ChunkIndex ✓')

In [ ]:
class Generator:
    def __init__(self, model: str = 'claude-sonnet-4-6', max_tokens: int = 256):
        self.client = anthropic.Anthropic()
        self.model = model
        self.max_tokens = max_tokens

    def generate(self, query: str, chunks: Optional[List[Chunk]] = None) -> tuple:
        if chunks:
            context = '\n\n'.join([f'[{i+1}] {c.text}' for i, c in enumerate(chunks)])
            prompt = f'Answer based on context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
        else:
            prompt = f'Answer from memory:\n\nQuestion: {query}\nAnswer:'
        resp = self.client.messages.create(
            model=self.model, max_tokens=self.max_tokens,
            messages=[{'role': 'user', 'content': prompt}]
        )
        tokens = resp.usage.input_tokens + resp.usage.output_tokens
        return resp.content[0].text.strip(), tokens

    def generate_candidate(self, query: str) -> str:
        """Pre-retrieval Active Recall: ask the model what it already 'remembers' before retrieving."""
        prompt = f'Before looking anything up, write a brief candidate answer from memory.\n\nQuestion: {query}\nCandidate answer:'
        resp = self.client.messages.create(
            model=self.model, max_tokens=128,
            messages=[{'role': 'user', 'content': prompt}]
        )
        return resp.content[0].text.strip()

print('Generator ✓')

## 3. EbbRAG (full combined system)

Combines SR-weighted retrieval (`use_sr`) and the pre-retrieval Active Recall step (`use_ar`). Default mixing weights: `alpha1=0.6` (semantic similarity), `alpha2=0.2` (SR weight), `alpha3=0.2` (AR-candidate similarity).

In [ ]:
class EbbRAG:
    def __init__(self, index: ChunkIndex, embedder: Embedder, generator: Generator,
                 k: int = 5, use_sr: bool = True, use_ar: bool = True,
                 alpha1: float = 0.6, alpha2: float = 0.2, alpha3: float = 0.2):
        self.index = index
        self.embedder = embedder
        self.generator = generator
        self.k = k
        self.use_sr = use_sr
        self.use_ar = use_ar
        self.alpha1 = alpha1
        self.alpha2 = alpha2 if use_sr else 0.0
        self.alpha3 = alpha3 if use_ar else 0.0

    def query(self, query: str, query_id: str = 'q0',
              ground_truth: Optional[str] = None,
              t_now: Optional[float] = None) -> QueryResult:
        start = time.time()
        t_now = t_now if t_now is not None else time.time()
        total_tokens = 0

        candidate_emb = None
        if self.use_ar:
            candidate = self.generator.generate_candidate(query)
            candidate_emb = self.embedder.embed(candidate)

        query_emb = self.embedder.embed(query)
        results = self.index.search(
            query_emb, k=self.k,
            t_now=t_now if self.use_sr else None,
            alpha1=self.alpha1, alpha2=self.alpha2,
            candidate_emb=candidate_emb, alpha3=self.alpha3,
        )
        chunks = [c for c, _ in results]
        scores = [s for _, s in results]

        if self.use_sr:
            for c in chunks:
                self.index.update_sr(c, t_now)

        final_answer, tok = self.generator.generate(query, chunks=chunks)
        total_tokens += tok

        em = int(ground_truth.lower().strip() in final_answer.lower()) if ground_truth else None
        return QueryResult(
            query_id=query_id, query=query,
            retrieved_chunks=chunks, retrieval_scores=scores,
            parametric_answer=None, final_answer=final_answer,
            latency_ms=(time.time() - start) * 1000,
            tokens_used=total_tokens, em=em
        )

print('EbbRAG ✓')

## 4. Ablation sweep harness

In [ ]:
def evaluate(system, eval_set, t_now=None):
    ems = []
    for ex in eval_set:
        r = system.query(ex['question'], query_id=ex['query_id'],
                          ground_truth=ex['answers'][0] if ex['answers'] else None,
                          t_now=t_now)
        if r.em is not None:
            ems.append(r.em)
    return float(np.mean(ems)) if ems else 0.0

In [ ]:
def run_lambda_sweep(index_factory, embedder, generator, eval_set, k=5,
                      lambdas=(0.01, 0.05, 0.1, 0.5)):
    """Sweep the SR decay rate lambda; everything else fixed."""
    rows = []
    for lam in tqdm(lambdas):
        index = index_factory(lambda_decay=lam)
        system = EbbRAG(index=index, embedder=embedder, generator=generator, k=k,
                         use_sr=True, use_ar=True)
        em = evaluate(system, eval_set)
        rows.append({'lambda_decay': lam, 'em': em})
    df = pd.DataFrame(rows)
    df.to_csv('results/ablations/lambda_sweep.csv', index=False)
    return df

In [ ]:
def run_alpha3_sweep(index, embedder, generator, eval_set, k=5,
                      alpha3_values=(0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0)):
    """Sweep the Active Recall mixing weight alpha3; everything else fixed."""
    rows = []
    for a3 in tqdm(alpha3_values):
        system = EbbRAG(index=index, embedder=embedder, generator=generator, k=k,
                         use_sr=True, use_ar=(a3 > 0), alpha3=a3)
        em = evaluate(system, eval_set)
        rows.append({'alpha3': a3, 'em': em})
    df = pd.DataFrame(rows)
    df.to_csv('results/ablations/alpha3_sweep.csv', index=False)
    return df

In [ ]:
def run_k_sweep(index, embedder, generator, eval_set, ks=(1, 3, 5, 10, 20)):
    """Sweep retrieval depth k; everything else fixed."""
    rows = []
    for k in tqdm(ks):
        system = EbbRAG(index=index, embedder=embedder, generator=generator, k=k,
                         use_sr=True, use_ar=True)
        em = evaluate(system, eval_set)
        rows.append({'k': k, 'em': em})
    df = pd.DataFrame(rows)
    df.to_csv('results/ablations/k_sweep.csv', index=False)
    return df

## 5. Plotting

In [ ]:
def plot_lambda_sweep(df):
    plt.figure(figsize=(5, 4))
    plt.plot(df['lambda_decay'], df['em'], marker='o')
    plt.xscale('log')
    plt.xlabel('lambda_decay')
    plt.ylabel('Exact Match')
    plt.title('EbbRAG: EM vs. decay rate (lambda)')
    plt.tight_layout()
    plt.savefig('results/ablations/lambda_sweep.png', dpi=150)
    plt.show()

def plot_alpha3_sweep(df):
    plt.figure(figsize=(5, 4))
    plt.plot(df['alpha3'], df['em'], marker='o', color='darkorange')
    plt.xlabel('alpha3 (Active Recall weight)')
    plt.ylabel('Exact Match')
    plt.title('EbbRAG: EM vs. AR mixing weight (alpha3)')
    plt.tight_layout()
    plt.savefig('results/ablations/alpha3_sweep.png', dpi=150)
    plt.show()

print('Plotting functions ✓')

## 6. Run ablations

> Run after `EbbRAG_04_data.ipynb` has produced `data/nq_dev_linked.jsonl` and the wiki chunk index.

In [ ]:
# eval_set = [json.loads(l) for l in open('data/nq_dev_linked.jsonl')][:200]
# embedder = Embedder()
# generator = Generator()
#
# def index_factory(lambda_decay=0.05):
#     idx = ChunkIndex.load('cache/wiki_index.pkl')
#     idx.lambda_decay = lambda_decay
#     return idx
#
# lambda_df = run_lambda_sweep(index_factory, embedder, generator, eval_set)
# plot_lambda_sweep(lambda_df)
#
# base_index = index_factory()
# alpha3_df = run_alpha3_sweep(base_index, embedder, generator, eval_set)
# plot_alpha3_sweep(alpha3_df)
#
# k_df = run_k_sweep(base_index, embedder, generator, eval_set)

print('Ablation harness ready ✓ (uncomment cell to execute full sweeps)')